<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.3：高阶函数
**上一步：[插曲：Chisel 标准库](3.2_interlude.ipynb)**<br>
**下一步：[函数式编程](3.4_functional_programming.ipynb)**

## 动机
上一模块中那些讨厌的 `for` 循环非常冗长，并且违背了函数式编程的目的！在本模块中，您的生成器将变得更加函数化。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 两个 FIR 滤波器的故事 <a name="compact-fir"></a>
在上一模块中，FIR 滤波器的卷积部分是这样写的：

```scala
val muls = Wire(Vec(length, UInt(8.W)))
for(i <- 0 until length) {
  if(i == 0) muls(i) := io.in * io.consts(i)
  else       muls(i) := regs(i - 1) * io.consts(i)
}

val scan = Wire(Vec(length, UInt(8.W)))
for(i <- 0 until length) {
  if(i == 0) scan(i) := muls(i)
  else scan(i) := muls(i) + scan(i - 1)
}

io.out := scan(length - 1)
```

回顾一下，其思想是将 `io.in` 的每个元素与 `io.consts` 的相应元素相乘，并将其存储在 `muls` 中。
然后，将 `muls` 中的元素累加到 `scan` 中，其中 `scan(0) = muls(0)`，`scan(1) = scan(0) + muls(1) = muls(0) + muls(1)`，通常 `scan(n) = scan(n-1) + muls(n) = muls(0) + ... + muls(n-1) + muls(n)`。
将 `scan` 中的最后一个元素（等于所有 `muls` 的总和）赋给 `io.out`。

然而，对于一个可能被认为非常简单的操作来说，这非常冗长。事实上，所有这些都可以写在一行中：

```scala
io.out := (taps zip io.consts).map { case (a, b) => a * b }.reduce(_ + _)
```

它在做什么？！让我们分解一下：
- 假设 `taps` 是所有样本的列表，其中 `taps(0) = io.in`，`taps(1) = regs(0)` 等。
- `(taps zip io.consts)` 获取两个列表 `taps` 和 `io.consts`，并将它们组合成一个列表，其中每个元素是相应位置输入的元素的元组。具体来说，其值为 `[(taps(0), io.consts(0)), (taps(1), io.consts(1)), ..., (taps(n), io.consts(n))]`。请记住，句点是可选的，因此这等效于 `(taps.zip(io.consts))`。
- `.map { case (a, b) => a * b }` 将匿名函数（接受一个包含两个元素的元组并返回它们的乘积）应用于列表的元素，并返回结果。在这种情况下，结果等效于详细示例中的 `muls`，其值为 `[taps(0) * io.consts(0), taps(1) * io.consts(1), ..., taps(n) * io.consts(n)]`。您将在下一个模块中重新学习匿名函数。现在，只需学习此语法即可。
- 最后，`.reduce(_ + _)` 也将函数（元素相加）应用于列表的元素。但是，它接受两个参数：第一个是当前累加值，第二个是列表元素（在第一次迭代中，它只是将前两个元素相加）。这些由括号中的两个下划线给出。假设从左到右遍历，结果将是 `(((muls(0) + muls(1)) + muls(2)) + ...) + muls(n)`，其中嵌套更深的括号的结果首先计算。这是卷积的输出。

---
# 函数作为参数
形式上，像 `map` 和 `reduce` 这样的函数称为_高阶函数_：它们是接受函数作为参数的函数。
事实证明（并且希望，正如您从上面的示例中看到的那样），这些是非常强大的结构，它们封装了一个通用的计算模式，使您能够专注于应用程序逻辑而不是流控制，并产生非常简洁的代码。

## 指定函数的不同方法
您可能已经注意到，在上面的示例中，有两种指定函数的方法：
- 对于每个参数仅被引用一次的函数，您*可能*能够使用下划线 (`_`) 来引用每个参数。在上面的示例中，`reduce` 参数函数接受两个参数，可以指定为 `_ + _`。虽然方便，但这受到一组额外的晦涩规则的约束，因此如果不起作用，请尝试：
- 显式指定输入参数列表。reduce 可以显式编写为 `(a, b) => a + b`，其一般形式是将参数列表放在括号中，后跟 `=>`，然后是引用这些参数的函数体。
- 当需要元组解包时，使用 `case` 语句，如 `case (a, b) => a * b`。它接受一个参数，一个包含两个元素的元组，并将其解包到变量 `a` 和 `b` 中，然后可以在函数体中使用它们。

## Scala 实践
在最后一个模块中，我们已经了解了 Scala 集合 API 中的主要类，例如 `List`。
这些高阶函数是这些 API 的一部分——事实上，上面的示例在 `List` 上使用了 `map` 和 `reduce` API。
在本节中，我们将通过示例和练习来熟悉这些方法。
在这些示例中，为了简单和清晰起见，我们将操作 Scala 数字 (`Int`)，但由于 Chisel 运算符的行为类似，因此这些概念应该可以推广。

<span style="color:blue">**示例：Map**</span><br>
`List[A].map` 的类型签名为 `map[B](f: (A) ⇒ B): List[B]`。您将在后续模块中了解有关类型的更多信息。现在，将类型 A 和 B 视为 `Int` 或 `UInt`，这意味着它们可以是软件或硬件类型。

通俗地说，它接受一个类型为 `(f: (A) ⇒ B)` 的参数，或者一个接受一个类型为 `A`（与输入列表的元素类型相同）的参数并返回一个类型为 `B`（可以是任何类型）的值的函数。然后，`map` 返回一个类型为 `B`（参数函数的返回类型）的新列表。

由于我们已经在 FIR 示例中解释了 List 的行为，因此让我们直接进入示例和练习：

In [ ]:
println(List(1, 2, 3, 4).map(x => x + 1))  // 函数中显式参数列表
println(List(1, 2, 3, 4).map(_ + 1))  // 等效于上面，但使用隐式参数
println(List(1, 2, 3, 4).map(_.toString + "a"))  // 输出元素类型可以与输入元素类型不同

println(List((1, 5), (2, 6), (3, 7), (4, 8)).map { case (x, y) => x*y })  // 这会解包一个元组，注意使用花括号

// 相关：Scala 有一种用于构造连续数字列表的语法
println(0 to 10)  // to 是包含性的，端点是结果的一部分
println(0 until 10)  // until 在末尾是排除性的，端点不是结果的一部分

// 这些在很大程度上表现得像列表，并且可以用于生成索引：
val myList = List("a", "b", "c", "d")
println((0 until 4).map(myList(_)))

<span style="color:red">**练习：Map**</span><br><a name="map-exercise"></a>

In [ ]:
// 现在你来试试：
// 填空 (???)，使其将输入列表的元素加倍。
// 这应该返回：List(2, 4, 6, 8)
println(List(1, 2, 3, 4).map(???))

<span style="color:blue">**示例：zipWithIndex**</span><br>
`List.zipWithIndex` 的类型签名为 `zipWithIndex: List[(A, Int)]`。

它不接受任何参数，但返回一个列表，其中每个元素都是原始元素和索引（第一个索引为零）的元组。
因此 `List("a", "b", "c", "d").zipWithIndex` 将返回 `List(("a", 0), ("b", 1), ("c", 2), ("d", 3))`

当在某些操作中需要元素索引时，这很有用。

由于这非常简单，我们只提供一些示例：

In [ ]:
println(List(1, 2, 3, 4).zipWithIndex)  // 注意索引从零开始
println(List("a", "b", "c", "d").zipWithIndex)
println(List(("a", "b"), ("c", "d"), ("e", "f"), ("g", "h")).zipWithIndex)  // 元组嵌套

<span style="color:blue">**示例：Reduce**</span><br>
`List[A].map` 的类型签名类似于 `reduce(op: (A, A) ⇒ A): A`。（实际上它更宽松，`A` 只需要是列表类型的超类型，但我们在这里不讨论该语法）

正如上面已经解释过的，这里有一些示例：

In [ ]:
println(List(1, 2, 3, 4).reduce((a, b) => a + b))  // 返回所有元素的总和
println(List(1, 2, 3, 4).reduce(_ * _))  // 返回所有元素的乘积
println(List(1, 2, 3, 4).map(_ + 1).reduce(_ + _))  // 您可以将 reduce 链接到 map 的结果上

In [ ]:
// 重要提示：reduce 在空列表上会失败
println(List[Int]().reduce(_ * _))

<span style="color:red">**练习：Reduce**</span><br><a name="reduce-exercise"></a>

In [ ]:
// 现在你来试试：
// 填空 (???)，使其返回输入列表元素乘积的两倍。
// 这应该返回：(1*2)*(2*2)*(3*2)*(4*2) = 384
println(List(1, 2, 3, 4).map(???).reduce(???))

<span style="color:blue">**示例：Fold**</span><br>
`List[A].fold` 与 reduce 非常相似，只是您可以指定初始累积值。
它的类型签名类似于 `fold(z: A)(op: (A, A) ⇒ A): A`。（与 `reduce` 类似，`A` 的类型也更宽松）

值得注意的是，它接受两个参数列表，第一个 (`z`) 是初始值，第二个是累积函数。
与 `reduce` 不同，它在空列表上不会失败，而是直接返回初始值。

以下是一些示例：

In [ ]:
println(List(1, 2, 3, 4).fold(0)(_ + _))  // 等效于使用 reduce 求和
println(List(1, 2, 3, 4).fold(1)(_ + _))  // 与上面类似，但累积从 1 开始
println(List().fold(1)(_ + _))  // 与 reduce 不同，在空输入上不会失败

<span style="color:red">**练习：Fold**</span><br><a name="fold-exercise"></a>

In [ ]:
// 现在你来试试：
// 填空 (???)，使其返回输入列表元素乘积的两倍。
// 这应该返回：2*(1*2*3*4) = 48
// 注意：除非需要空列表容错，否则 reduce 在这里更合适。
println(List(1, 2, 3, 4).fold(???)(???))

<span style="color:red">**练习：解耦仲裁器**</span><br>
现在让我们把所有东西整合到一个练习中。

对于这个例子，我们将构建一个解耦仲裁器：一个具有 _n_ 个解耦输入和一个解耦输出的模块。
仲裁器选择第一个有效的通道并将其转发到输出。

一些提示：
- 架构上：
  - 如果任何输入有效，则 `io.out.valid` 为真
  - 考虑使用一个内部线来表示所选通道
  - 每个输入的 `ready` 为真，条件是输出准备就绪，并且该通道被选中（这确实会组合地耦合 ready 和 valid，但我们暂时忽略它……）
- 这些结构可能会有所帮助：
  - `map`，尤其适用于返回子元素的 Vec，例如 `io.in.map(_.valid)` 返回输入 Bundle 的有效信号列表
  - `PriorityMux(List[Bool, Bits])`，它接受一个有效信号和位的列表，返回第一个有效的元素
  - Vec 上的动态索引，通过使用 UInt 进行索引，例如 `io.in(0.U)`

In [ ]:
class MyRoutingArbiter(numChannels: Int) extends Module {
  val io = IO(new Bundle {
    val in = Vec(numChannels, Flipped(Decoupled(UInt(8.W))))
    val out = Decoupled(UInt(8.W))
  } )

  // 你的代码如下
  ???
}

test(new MyRoutingArbiter(4)) { c =>
    // 验证计算是否正确
    // 设置输入默认值
    for(i <- 0 until 4) {
        c.io.in(i).valid.poke(false.B)
        c.io.in(i).bits.poke(i.U)
        c.io.out.ready.poke(true.B)
    }

    c.io.out.valid.expect(false.B)

    // 检查带反压的单个输入有效行为
    for (i <- 0 until 4) {
        c.io.in(i).valid.poke(true.B)
        c.io.out.valid.expect(true.B)
        c.io.out.bits.expect(i.U)

        c.io.out.ready.poke(false.B)
        c.io.in(i).ready.expect(false.B)

        c.io.out.ready.poke(true.B)
        c.io.in(i).valid.poke(false.B)
    }

    // 带反压的多个输入就绪行为的基本检查
    c.io.in(1).valid.poke(true.B)
    c.io.in(2).valid.poke(true.B)
    c.io.out.bits.expect(1.U)
    c.io.in(1).ready.expect(true.B)
    c.io.in(0).ready.expect(false.B)

    c.io.out.ready.poke(false.B)
    c.io.in(1).ready.expect(false.B)
}

println("成功！！") // Scala 代码：如果我们到达这里，我们的测试通过了！

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
class MyRoutingArbiter(numChannels: Int) extends Module {
  val io = IO(new Bundle {
    val in = Vec(numChannels, Flipped(Decoupled(UInt(8.W))))
    val out = Decoupled(UInt(8.W))
  } )

  // 你的代码如下
  io.out.valid := io.in.map(\_.valid).reduce(\_ || \_)
  val channel = PriorityMux(
    io.in.map(\_.valid).zipWithIndex.map { case (valid, index) => (valid, index.U) }
  )
  io.out.bits := io.in(channel).bits
  io.in.map(\_.ready).zipWithIndex.foreach { case (ready, index) =>
    ready := io.out.ready && channel === index.U
  }
}
</pre></article></div></section></div>

---
# 您已完成！

[返回顶部。](#top)